<a href="https://colab.research.google.com/github/jkshakthi999/Context-Aware-Scene-Colorizer/blob/main/task4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU. Go to Runtime > Change runtime type > T4 GPU, then run this cell again.")

!pip install -q gradio scikit-image
print("Libraries installed.")

PyTorch: 2.14.1+cpu
GPU available: False
Libraries installed.


In [2]:
import os, io, glob, time, shutil, random, traceback
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms as T
from PIL import Image, ImageOps
from skimage import color
from scipy.ndimage import uniform_filter
import gradio as gr

# ------------------------- Google Drive -------------------------
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = "/content/drive/MyDrive/scene_colorization"      # photos + trained model are saved here
except Exception as e:
    print("Could not mount Google Drive (", e, ")\nUsing temporary Colab storage - files will be lost when the session ends.")
    DATA_ROOT = "/content/scene_colorization"

# ------------------------- scene types (folder names) -------------------------
# You may edit this list BEFORE the first training. If you change it later, train again.
SCENES = ["cityscape", "forest", "mountain", "coast", "countryside"]
S2I = {s: i for i, s in enumerate(SCENES)}
SCENE_LABELS = {"cityscape": "🏙 Cityscape", "forest": "🌲 Forest", "mountain": "⛰ Mountain",
                "coast": "🌊 Coast / water", "countryside": "🌾 Countryside"}
for s in SCENES:
    SCENE_LABELS.setdefault(s, s)

# ------------------------- settings -------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 256        # training size (divisible by 32). 320 gives finer detail but trains slower
INFER_SIDE = 256      # longer side used by the model when colorizing
MAX_SIDE = 2048       # very large uploads are shrunk to this
MIN_SAT = 0.03        # photos with lower average saturation are treated as black & white and skipped in training
BATCH_SIZE = 16
LR = 3e-4
COLOR_W = 10.0        # weight of the colour (ab) loss
CLS_W = 0.5           # weight of the scene-classification loss
PERC_W = 1.0          # weight of the perceptual (VGG16) loss; 0 = off (faster, no 0.5 GB download)
NUM_WORKERS = 2
PRETRAINED = True     # ImageNet weights for the encoder (downloaded automatically)

EXTS = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp")
CACHE_ROOT = "/content/scene_cache"                       # resized copies for fast training (temporary)
CKPT = os.path.join(DATA_ROOT, "scene_colorizer.pt")      # trained model

for s in SCENES:
    os.makedirs(os.path.join(DATA_ROOT, s), exist_ok=True)
random.seed(0); np.random.seed(0); torch.manual_seed(0)
print("Device:", device)
print("Data folder:", DATA_ROOT)
print("Scene types:", SCENES)

Mounted at /content/drive
Device: cpu
Data folder: /content/drive/MyDrive/scene_colorization
Scene types: ['cityscape', 'forest', 'mountain', 'coast', 'countryside']


In [3]:
def prepare_scene(s):
    # reads photos of one scene type, skips bad / black & white ones, saves resized copies for fast training
    out_dir = os.path.join(CACHE_ROOT, s)
    os.makedirs(out_dir, exist_ok=True)
    good, bad = [], []
    for f in sorted(glob.glob(os.path.join(DATA_ROOT, s, "*"))):
        if not f.lower().endswith(EXTS):
            continue
        try:
            with Image.open(f) as im:
                try:
                    im.draft("RGB", (1024, 1024))
                except Exception:
                    pass
                im = ImageOps.exif_transpose(im).convert("RGB")
            w, h = im.size
            if min(w, h) < 64:
                bad.append((os.path.basename(f), "too small"))
                continue
            sat = np.asarray(im.resize((64, 64)).convert("HSV"))[..., 1].mean() / 255.0
            if sat < MIN_SAT:
                bad.append((os.path.basename(f), "looks black & white (training photos must be colour)"))
                continue
            k = 320.0 / min(w, h)
            if k < 1.0:
                im = im.resize((max(1, int(round(w * k))), max(1, int(round(h * k)))), Image.LANCZOS)
            dst = os.path.join(out_dir, f"{s}_{len(good):05d}.jpg")
            im.save(dst, quality=95)
            good.append(dst)
        except Exception as e:
            bad.append((os.path.basename(f), f"unreadable: {e}"))
    return good, bad


class SceneDataset(Dataset):
    def __init__(self, items, train=True):
        self.items = items            # list of (path, scene_id)
        if train:
            self.tf = T.Compose([T.RandomResizedCrop(IMG_SIZE, scale=(0.3, 1.0)), T.RandomHorizontalFlip()])
        else:
            self.tf = T.Compose([T.Resize(IMG_SIZE), T.CenterCrop(IMG_SIZE)])

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        path, sid = self.items[i]
        with Image.open(path) as im:
            img = self.tf(im.convert("RGB"))
        lab = color.rgb2lab(np.asarray(img)).astype("float32")
        L = torch.from_numpy(lab[..., 0:1] / 50.0 - 1.0).permute(2, 0, 1)     # lightness -> [-1, 1]
        ab = torch.from_numpy(lab[..., 1:] / 128.0).permute(2, 0, 1)          # colour    -> ~[-1, 1]
        return L, ab, sid


# ---------------- differentiable Lab -> sRGB (used by the perceptual loss) ----------------
def lab_to_rgb_t(L, ab):
    # L: B x 1 x H x W in [-1, 1] ; ab: B x 2 x H x W in ~[-1, 1]  ->  sRGB B x 3 x H x W in [0, 1]
    Lc = (L + 1.0) * 50.0
    a = ab[:, 0:1] * 128.0
    b = ab[:, 1:2] * 128.0
    fy = (Lc + 16.0) / 116.0
    fx = fy + a / 500.0
    fz = fy - b / 200.0

    def finv(t):
        return torch.where(t > 6.0 / 29.0, t ** 3, 3.0 * (6.0 / 29.0) ** 2 * (t - 4.0 / 29.0))

    X, Y, Z = 0.95047 * finv(fx), 1.0 * finv(fy), 1.08883 * finv(fz)
    r = 3.2404542 * X - 1.5371385 * Y - 0.4985314 * Z
    g = -0.9692660 * X + 1.8760108 * Y + 0.0415560 * Z
    bl = 0.0556434 * X - 0.2040259 * Y + 1.0572252 * Z
    lin = torch.cat([r, g, bl], dim=1).clamp(0.0, 1.0)
    return torch.where(lin > 0.0031308, 1.055 * lin.clamp(min=1e-8) ** (1.0 / 2.4) - 0.055, 12.92 * lin)


# ---------------- model ----------------
def pos_enc_2d(h, w, dim, device):
    # sinusoidal 2-D positional encoding (dim divisible by 4) -> (h*w) x dim
    d = dim // 2

    def enc1d(n):
        pos = torch.arange(n, device=device, dtype=torch.float32)[:, None]
        i = torch.arange(0, d, 2, device=device, dtype=torch.float32)[None, :]
        ang = pos / (10000.0 ** (i / d))
        e = torch.zeros(n, d, device=device)
        e[:, 0::2] = torch.sin(ang)
        e[:, 1::2] = torch.cos(ang)
        return e

    ey = enc1d(h)[:, None, :].expand(h, w, d)
    ex = enc1d(w)[None, :, :].expand(h, w, d)
    return torch.cat([ey, ex], dim=-1).reshape(h * w, dim)


class ContextBlock(nn.Module):
    # global self-attention: every location can look at every other location of the scene
    def __init__(self, dim, heads=8):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, dim * 2), nn.GELU(), nn.Linear(dim * 2, dim))
        self.gamma = nn.Parameter(torch.zeros(1))        # starts as identity, learns how much context to mix in

    def forward(self, x):
        B, C, H, W = x.shape
        t = x.flatten(2).transpose(1, 2)                                  # B x HW x C
        t = t + pos_enc_2d(H, W, C, x.device).to(t.dtype)[None]
        h = self.norm1(t)
        a, _ = self.attn(h, h, h, need_weights=False)
        delta = a + self.mlp(self.norm2(t + a))
        return x + self.gamma * delta.transpose(1, 2).reshape(B, C, H, W).to(x.dtype)


class FiLM(nn.Module):
    # scales and shifts feature maps depending on the scene type
    def __init__(self, emb_dim, ch):
        super().__init__()
        self.fc = nn.Linear(emb_dim, ch * 2)
        nn.init.zeros_(self.fc.weight)
        nn.init.zeros_(self.fc.bias)

    def forward(self, x, e):
        g, b = self.fc(e).chunk(2, dim=1)
        return x * (1 + g[:, :, None, None]) + b[:, :, None, None]


class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, emb_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(True))
        self.film = FiLM(emb_dim, out_ch)

    def forward(self, x, skip, e):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = self.conv(torch.cat([x, skip], dim=1))
        return self.film(x, e)


class SceneColorizer(nn.Module):
    # grayscale (L) + scene  ->  colour (ab) ;  grayscale (L)  ->  scene logits
    def __init__(self, n_scenes, emb_dim=64, pretrained=True):
        super().__init__()
        r = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None)
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu)
        self.pool = r.maxpool
        self.l1, self.l2, self.l3, self.l4 = r.layer1, r.layer2, r.layer3, r.layer4
        self.ctx3 = ContextBlock(256)
        self.ctx4 = ContextBlock(512)
        self.emb = nn.Embedding(n_scenes, emb_dim)
        self.film4 = FiLM(emb_dim, 512)
        self.up3 = UpBlock(512, 256, 256, emb_dim)
        self.up2 = UpBlock(256, 128, 128, emb_dim)
        self.up1 = UpBlock(128, 64, 64, emb_dim)
        self.up0 = UpBlock(64, 64, 64, emb_dim)
        self.head = nn.Sequential(nn.Conv2d(64 + 1, 32, 3, padding=1), nn.ReLU(True),
                                  nn.Conv2d(32, 2, 1), nn.Tanh())
        self.cls = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(512, n_scenes))

    def encode(self, L):
        x = L.repeat(1, 3, 1, 1)
        s0 = self.stem(x)
        s1 = self.l1(self.pool(s0))
        s2 = self.l2(s1)
        s3 = self.ctx3(self.l3(s2))
        s4 = self.ctx4(self.l4(s3))
        return s0, s1, s2, s3, s4

    def scene_logits(self, L):
        return self.cls(self.encode(L)[4])

    def forward(self, L, scene):
        # scene: LongTensor of scene ids (B)  or  FloatTensor of scene probabilities (B x n_scenes) for a blended style
        s0, s1, s2, s3, s4 = self.encode(L)
        logits = self.cls(s4)
        e = self.emb(scene) if scene.dtype in (torch.long, torch.int64) else scene @ self.emb.weight
        d = self.up3(self.film4(s4, e), s3, e)
        d = self.up2(d, s2, e)
        d = self.up1(d, s1, e)
        d = self.up0(d, s0, e)
        d = F.interpolate(d, size=L.shape[-2:], mode="bilinear", align_corners=False)
        return self.head(torch.cat([d, L], dim=1)), logits


def guided_filter(I, p, r, eps):
    # edge-aware smoothing of p guided by I (He et al.): colours follow the brightness edges of the photo
    size = 2 * r + 1
    mean_I, mean_p = uniform_filter(I, size), uniform_filter(p, size)
    var_I = uniform_filter(I * I, size) - mean_I * mean_I
    cov_Ip = uniform_filter(I * p, size) - mean_I * mean_p
    a = cov_Ip / (var_I + eps)
    b = mean_p - a * mean_I
    return uniform_filter(a, size) * I + uniform_filter(b, size)

print("Dataset and model code ready.")

Dataset and model code ready.


In [4]:
model = None            # trained model (filled by training, or loaded from Drive in Cell 5)
ACTIVE_SCENES = []      # scene types the model was trained on


def count_photos(s):
    folder = os.path.join(DATA_ROOT, s)
    if not os.path.isdir(folder):
        return 0
    return len([f for f in os.listdir(folder) if f.lower().endswith(EXTS)])


def dataset_status():
    rows = ["| Scene type | Photos saved in Drive |", "|---|---|"]
    for s in SCENES:
        n = count_photos(s)
        flag = "" if n >= 20 else ("  (need more, 20+ recommended)" if n > 0 else "  (none yet)")
        rows.append(f"| {SCENE_LABELS[s]} | {n}{flag} |")
    return "\n".join(rows)


def save_uploads(scene, uploaded):
    if not uploaded:
        return "No photos selected. Click the upload box, choose photos, then click Save.", dataset_status(), None
    folder = os.path.join(DATA_ROOT, scene)
    os.makedirs(folder, exist_ok=True)
    ok = bad = 0
    stamp = int(time.time())
    for i, f in enumerate(uploaded):
        path = f if isinstance(f, str) else getattr(f, "name", None)
        if path is None or os.path.splitext(path)[1].lower() not in EXTS:
            bad += 1
            continue
        try:
            with Image.open(path) as im:
                im.verify()
        except Exception:
            bad += 1
            continue
        shutil.copy(path, os.path.join(folder, f"up_{stamp}_{i:04d}{os.path.splitext(path)[1].lower()}"))
        ok += 1
    msg = f"Saved **{ok}** photo(s) to **{SCENE_LABELS[scene]}**." + (f" Skipped {bad} (not valid images)." if bad else "")
    return msg + " Choose another scene type to upload more, or go to the Train Model tab.", dataset_status(), None


def train_model(epochs):
    global model, ACTIVE_SCENES
    epochs = int(epochs)
    log = []

    def emit(msg):
        log.append(msg)
        return "\n".join(log)

    yield emit("Preparing dataset ...")
    if os.path.exists(CACHE_ROOT):
        shutil.rmtree(CACHE_ROOT)
    items, counts = [], {}
    for s in SCENES:
        good, bad = prepare_scene(s)
        counts[s] = len(good)
        yield emit(f"  {s:12s} usable photos: {len(good):4d}   skipped: {len(bad)}")
        for name, why in bad[:3]:
            yield emit(f"      skipped {name}: {why}")
        if 0 < len(good) < 20:
            yield emit(f"    WARNING: only {len(good)} photos for {s}; results for this scene type will be weak.")
        items += [(f, S2I[s]) for f in good]
    if len(items) < 8:
        raise gr.Error("Fewer than 8 usable colour photos in total. Upload more colour photos in the first tab.")
    active = [s for s in SCENES if counts[s] > 0]
    if len(active) < 2:
        yield emit("NOTE: only one scene type has photos, so scene detection cannot learn to tell scenes apart.")
    yield emit("Scene types that will be learned: " + ", ".join(active))

    random.shuffle(items)
    n_val = max(2, int(0.1 * len(items)))
    val_items, train_items = items[:n_val], items[n_val:]
    train_ds = SceneDataset(train_items, train=True)
    val_ds = SceneDataset(val_items, train=False)
    per_scene = {}
    for _, sid in train_items:
        per_scene[sid] = per_scene.get(sid, 0) + 1
    weights = [1.0 / per_scene[sid] for _, sid in train_items]         # every scene type is sampled equally often
    bs = min(BATCH_SIZE, len(train_ds))
    sampler = WeightedRandomSampler(weights, num_samples=len(train_ds), replacement=True)
    train_dl = DataLoader(train_ds, batch_size=bs, sampler=sampler, num_workers=NUM_WORKERS, drop_last=len(train_ds) > bs)
    val_dl = DataLoader(val_ds, batch_size=bs, shuffle=False, num_workers=NUM_WORKERS)
    yield emit(f"Training photos: {len(train_ds)} | validation photos: {len(val_ds)} | batch size: {bs}")

    vgg = None
    if PERC_W > 0:
        try:
            yield emit("Loading VGG16 for the perceptual loss (first time downloads about 0.5 GB) ...")
            vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).features[:16].to(device).eval()
            for q in vgg.parameters():
                q.requires_grad_(False)
        except Exception as e:
            vgg = None
            yield emit(f"  perceptual loss disabled ({e})")
    vmean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
    vstd = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

    yield emit("Building model (downloads pretrained weights the first time) ...")
    net = SceneColorizer(len(SCENES), pretrained=PRETRAINED).to(device)
    use_amp = (device == "cuda")
    opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=epochs, steps_per_epoch=len(train_dl))
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    color_loss = nn.SmoothL1Loss()
    cls_loss = nn.CrossEntropyLoss()
    best_val = float("inf")
    t0 = time.time()
    yield emit(f"Training for {epochs} epochs on {device} ...")

    for epoch in range(epochs):
        net.train()
        tr = 0.0
        for L, ab, sid in train_dl:
            L, ab, sid = L.to(device), ab.to(device), sid.to(device)
            with torch.autocast(device_type=device, enabled=use_amp):
                pred, logits = net(L, sid)
            pred = pred.float()
            loss = COLOR_W * color_loss(pred, ab) + CLS_W * cls_loss(logits.float(), sid)
            if vgg is not None:
                rgb_pred = lab_to_rgb_t(L, pred)
                with torch.no_grad():
                    rgb_true = lab_to_rgb_t(L, ab)
                with torch.autocast(device_type=device, enabled=use_amp):
                    f_pred = vgg((rgb_pred - vmean) / vstd)
                    with torch.no_grad():
                        f_true = vgg((rgb_true - vmean) / vstd)
                loss = loss + PERC_W * F.l1_loss(f_pred.float(), f_true.float())
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            tr += loss.item()
        tr /= len(train_dl)

        net.eval()
        v_col = v_cls = 0.0
        correct = total = 0
        with torch.no_grad():
            for L, ab, sid in val_dl:
                L, ab, sid = L.to(device), ab.to(device), sid.to(device)
                pred, logits = net(L, sid)
                v_col += color_loss(pred.float(), ab).item()
                v_cls += cls_loss(logits.float(), sid).item()
                correct += (logits.argmax(1) == sid).sum().item()
                total += sid.numel()
        v_col /= len(val_dl)
        v_cls /= len(val_dl)
        acc = correct / max(1, total)
        va = COLOR_W * v_col + CLS_W * v_cls

        tag = ""
        if va < best_val:
            best_val = va
            torch.save({"model": net.state_dict(), "scenes": SCENES, "active": active}, CKPT)
            tag = "   <- best so far, saved"
        yield emit(f"epoch {epoch + 1:02d}/{epochs}   train {tr:.3f}   val colour {v_col:.4f}   "
                   f"scene accuracy {acc * 100:4.1f}%   ({time.time() - t0:.0f}s){tag}")

    ck = torch.load(CKPT, map_location=device)
    net.load_state_dict(ck["model"])
    net.eval()

    stats = {s: [0, 0, 0.0] for s in SCENES}      # correct, total, colour error sum
    with torch.no_grad():
        for L, ab, sid in val_dl:
            pred, logits = net(L.to(device), sid.to(device))
            err = (pred.float().cpu() - ab).abs().mean(dim=(1, 2, 3))
            guess = logits.argmax(1).cpu()
            for k in range(len(sid)):
                name = SCENES[int(sid[k])]
                stats[name][1] += 1
                stats[name][0] += int(guess[k] == sid[k])
                stats[name][2] += float(err[k])
    yield emit("\nPer-scene results on validation photos (scene detection accuracy | colour error, lower is better):")
    for s in SCENES:
        c, n, e = stats[s]
        if n > 0:
            yield emit(f"  {s:12s} {100.0 * c / n:5.1f}%   {e / n:.4f}   ({n} photos)")

    model = net
    ACTIVE_SCENES = active
    yield emit("\nDONE. Model saved to Drive. Go to the 'Colorize' tab.")

print("Upload and training functions ready.")

Upload and training functions ready.


In [5]:
AUTO = "auto"


def _round32(v):
    return max(64, int(round(v / 32.0)) * 32)


def _prep(img):
    # PIL image -> model input tensor + full-resolution lightness
    img = ImageOps.exif_transpose(img).convert("RGB")
    W, H = img.size
    if max(W, H) > MAX_SIDE:
        k = MAX_SIDE / float(max(W, H))
        W, H = max(1, int(W * k)), max(1, int(H * k))
        img = img.resize((W, H), Image.LANCZOS)
    L_full = color.rgb2lab(np.asarray(img))[..., 0].astype(np.float32)        # 0..100 at full resolution
    k = INFER_SIDE / float(max(W, H))
    small = img.resize((_round32(W * k), _round32(H * k)), Image.BICUBIC)
    L = color.rgb2lab(np.asarray(small))[..., 0] / 50.0 - 1.0
    x = torch.from_numpy(L).float()[None, None].to(device)
    return x, L_full, H, W


def _check_ready():
    if model is None or not ACTIVE_SCENES:
        raise gr.Error("No trained model yet. Go to the 'Train Model' tab and click Start training first.")


def _scene_probs(x):
    # tensor 1 x n_scenes of probabilities (only over the scene types the model was trained on)
    lg = (model.scene_logits(x) + model.scene_logits(torch.flip(x, dims=[3]))) / 2.0      # flip test-time augmentation
    mask = torch.full_like(lg, -1e9)
    for s in ACTIVE_SCENES:
        mask[0, S2I[s]] = 0.0
    return torch.softmax(lg.float() + mask, dim=1)


@torch.no_grad()
def detect_scene(img):
    # returns {scene: probability} over the trained scene types
    _check_ready()
    model.eval()
    x, _, _, _ = _prep(img)
    p = _scene_probs(x)[0].cpu().numpy()
    return {s: float(p[S2I[s]]) for s in ACTIVE_SCENES}


@torch.no_grad()
def colorize(img, scene=AUTO, saturation=1.0, refine=0.7):
    # scene = AUTO (blend of detected scenes) or one scene name. refine: 0..1 strength of edge-aware colour refinement
    _check_ready()
    model.eval()
    x, L_full, H, W = _prep(img)
    if scene == AUTO or scene is None:
        cond = _scene_probs(x)
    else:
        if scene not in ACTIVE_SCENES:
            raise gr.Error(f"The model has no training photos for '{scene}'. Available: {', '.join(ACTIVE_SCENES)}")
        cond = torch.tensor([S2I[scene]], device=device)
    ab = model(x, cond)[0].float().cpu()                                        # 1 x 2 x h x w in [-1, 1]
    ab = F.interpolate(ab, size=(H, W), mode="bilinear", align_corners=False)[0].permute(1, 2, 0).numpy()
    if refine > 0:                                                              # snap colours to the photo's brightness edges
        guide = (L_full / 100.0).astype(np.float32)
        r = max(2, int(round(max(H, W) / 150.0)))
        smooth = np.stack([guided_filter(guide, np.ascontiguousarray(ab[..., k]).astype(np.float32), r, 1e-3)
                           for k in range(2)], axis=-1)
        ab = (1.0 - refine) * ab + refine * smooth
    ab = (ab * 128.0 * saturation).clip(-128, 127)
    lab = np.concatenate([L_full[..., None], ab], axis=-1)
    rgb = (color.lab2rgb(lab).clip(0, 1) * 255).astype(np.uint8)
    return Image.fromarray(rgb)


# load a previously trained model from Drive, if there is one
if os.path.exists(CKPT):
    try:
        ck = torch.load(CKPT, map_location=device)
        if ck["scenes"] == SCENES:
            model = SceneColorizer(len(SCENES), pretrained=False).to(device)
            model.load_state_dict(ck["model"])
            model.eval()
            ACTIVE_SCENES = ck.get("active", SCENES)
            print("Loaded saved model from Drive. Scene types:", ACTIVE_SCENES)
        else:
            print("Saved model uses a different scene list; please train again in the app.")
    except Exception as e:
        print("Could not load the saved model (", e, "). Please train again in the app.")
else:
    print("No saved model yet. Upload photos and train inside the app (next cell).")

No saved model yet. Upload photos and train inside the app (next cell).


In [ ]:
try:
    demo.close()          # close an older app if this cell is re-run
except Exception:
    pass

AUTO_LABEL = "🔎 Auto-detect scene (blends detected scenes)"


def scene_choices(names):
    return [(AUTO_LABEL, AUTO)] + [(SCENE_LABELS[s], s) for s in names]


def refresh_state():
    # shows the model status and the scene choices: Auto-detect + the scene types the model was trained on
    global model, ACTIVE_SCENES
    if model is None and os.path.exists(CKPT):
        try:
            ck = torch.load(CKPT, map_location=device)
            if ck["scenes"] == SCENES:
                m = SceneColorizer(len(SCENES), pretrained=False).to(device)
                m.load_state_dict(ck["model"])
                m.eval()
                model = m
                ACTIVE_SCENES = ck.get("active", SCENES)
        except Exception as e:
            print("Could not load saved model:", e)
    if model is None or not ACTIVE_SCENES:
        return ("⚠️ **No trained model yet.** Go to tab **2. Train Model** and click **Start training** "
                "(upload colour photos in tab 1 first).",
                gr.Radio(choices=scene_choices(SCENES), value=AUTO, label="Scene type"))
    return (f"✅ **Model ready.** Scene types: {', '.join(ACTIVE_SCENES)}",
            gr.Radio(choices=scene_choices(ACTIVE_SCENES), value=AUTO, label="Scene type (auto-detected, or choose manually)"))


def run_colorize(img, scene_choice, saturation, refine):
    if img is None:
        raise gr.Error("Please upload a photo first.")
    try:
        t0 = time.time()
        probs = detect_scene(img)
        top = max(probs, key=probs.get)
        if scene_choice == AUTO:
            note = (f"Scene understanding: mostly **{SCENE_LABELS[top]}** ({probs[top] * 100:.0f}%). "
                    f"Colours were chosen from a blend of the detected scenes.")
        else:
            note = (f"Manual scene: **{SCENE_LABELS[scene_choice]}**. "
                    f"(The model's own guess was {SCENE_LABELS[top]}, {probs[top] * 100:.0f}%.)")
        res = colorize(img, scene_choice, saturation, refine)
        orig = ImageOps.exif_transpose(img).convert("RGB")
        if orig.size != res.size:
            orig = orig.resize(res.size, Image.LANCZOS)
        label_probs = {SCENE_LABELS[s]: p for s, p in probs.items()}
        return res, orig, res, label_probs, note + f"  ({time.time() - t0:.1f}s)"
    except gr.Error:
        raise
    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f"{type(e).__name__}: {e}")


def compare_styles(img, saturation, refine):
    if img is None:
        raise gr.Error("Please upload a photo first.")
    try:
        probs = detect_scene(img)
        return [(colorize(img, s, saturation, refine), f"{SCENE_LABELS[s]}  ({probs[s] * 100:.0f}% match)")
                for s in ACTIVE_SCENES]
    except gr.Error:
        raise
    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f"{type(e).__name__}: {e}")


HEADER = (
    "<h1 style='text-align:center; margin-bottom:0'>🏙 Context-Aware Scene Colorizer</h1>"
    "<p style='text-align:center; margin-top:4px'><b>Understands the whole scene</b> &bull; <b>Global context attention</b> &bull; "
    "<b>Edge-aware colours</b> &bull; <b>Cityscapes, forests, mountains, coasts</b></p>"
)

with gr.Blocks(title="Context-Aware Scene Colorizer") as demo:
    gr.HTML(HEADER)
    with gr.Tabs():

        # ---------------- Tab 1: upload training photos ----------------
        with gr.Tab("1. Upload Training Photos"):
            gr.Markdown("Upload **real colour photos** of one scene type at a time. Select the scene type, choose the photos, "
                        "click **Save photos**, and repeat for each type. The model learns the natural colours of each kind of scene "
                        "(and how shadows, reflections and neighbouring objects relate). Use varied, sharp photos of complex scenes. "
                        "Aim for 100+ per scene type (20 minimum), and at least 2 scene types.")
            with gr.Row():
                with gr.Column():
                    up_scene = gr.Radio(choices=[(SCENE_LABELS[s], s) for s in SCENES], value=SCENES[0], label="Scene type of these photos")
                    up_files = gr.File(label="Choose colour photos", file_count="multiple", file_types=["image"])
                    up_btn = gr.Button("Save photos", variant="primary")
                    up_msg = gr.Markdown()
                with gr.Column():
                    ds_md = gr.Markdown(dataset_status())
                    ds_btn = gr.Button("Refresh counts")
            up_btn.click(save_uploads, inputs=[up_scene, up_files], outputs=[up_msg, ds_md, up_files])
            ds_btn.click(dataset_status, outputs=ds_md)

        # ---------------- Tab 2: train ----------------
        with gr.Tab("2. Train Model"):
            gr.Markdown("Click **Start training** after uploading photos. It takes roughly 10-30 minutes on a T4 GPU "
                        "(longer than the simpler projects because of the context blocks and the perceptual loss). "
                        "For a quick test, use 8 epochs. The best model is saved to Google Drive automatically.")
            epochs_sl = gr.Slider(5, 60, value=25, step=1, label="Epochs")
            train_btn = gr.Button("Start training", variant="primary")
            log_box = gr.Textbox(label="Training log", lines=16, max_lines=16, interactive=False)

        # ---------------- Tab 3: colorize ----------------
        with gr.Tab("3. Colorize"):
            model_md = gr.Markdown()
            with gr.Row():
                with gr.Column(scale=1):
                    inp = gr.Image(type="pil", label="📷 Upload B&W Photo", height=380)
                    scene_dd = gr.Radio(choices=scene_choices(SCENES), value=AUTO, label="Scene type (auto-detected, or choose manually)")
                    sat = gr.Slider(0.5, 2.0, value=1.2, step=0.1, label="Colour strength")
                    refine_sl = gr.Slider(0.0, 1.0, value=0.7, step=0.05, label="Edge-aware colour refinement")
                    btn = gr.Button("🎨 Colorize", variant="primary")
                    btn_all = gr.Button("Compare scene styles")
                    refresh_btn = gr.Button("Refresh model status")
                with gr.Column(scale=2):
                    status = gr.Markdown("Upload a photo and click **Colorize**. Leave *Auto-detect* on, or pick a scene type to override it.")
                    with gr.Tabs():
                        with gr.Tab("🌈 Colorized Result"):
                            out_main = gr.Image(type="pil", label="Colorized", height=520)
                        with gr.Tab("Side by Side Comparison"):
                            gr.Markdown("<p style='text-align:center'><b>Original (left) | Colorized (right)</b></p>")
                            with gr.Row():
                                out_orig = gr.Image(type="pil", label="Original", height=480)
                                out_col = gr.Image(type="pil", label="Colorized", height=480)
                        with gr.Tab("Scene Understanding"):
                            out_probs = gr.Label(label="How much the photo matches each scene type", num_top_classes=len(SCENES))
                        with gr.Tab("Compare Scene Styles"):
                            gallery = gr.Gallery(label="Same photo with each scene style", columns=3)

    train_btn.click(train_model, inputs=epochs_sl, outputs=log_box).then(refresh_state, None, [model_md, scene_dd])
    refresh_btn.click(refresh_state, None, [model_md, scene_dd])
    btn.click(run_colorize, inputs=[inp, scene_dd, sat, refine_sl], outputs=[out_main, out_orig, out_col, out_probs, status])
    btn_all.click(compare_styles, inputs=[inp, sat, refine_sl], outputs=gallery)
    demo.load(refresh_state, None, [model_md, scene_dd])

demo.queue()
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://84231be900c11f6fe8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/queueing.py", line 973, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<6 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/route_utils.py", line 418, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 2698, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 1975, in call_function
    prediction = await utils.async_iteration(iterator)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/gradio/utils.py", line 926, in async_iteratio

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:05<00:00, 101MB/s]


Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 106MB/s] 
/tmp/ipykernel_558/1024477033.py:70: UserWarning: Conversion from CIE-LAB, via XYZ to sRGB color space resulted in 435538 negative Z values that have been clipped to zero
  rgb = (color.lab2rgb(lab).clip(0, 1) * 255).astype(np.uint8)
/tmp/ipykernel_558/1024477033.py:70: UserWarning: Conversion from CIE-LAB, via XYZ to sRGB color space resulted in 47438 negative Z values that have been clipped to zero
  rgb = (color.lab2rgb(lab).clip(0, 1) * 255).astype(np.uint8)
